# EDA: 데이터 품질 문제 시각화

데이터 웨어하우스 구축 전 발견된 데이터 품질 문제를 시각화합니다.

**발견된 문제:**
1. 중복 데이터
2. 경과일수 컬럼의 이상값 (999999999)
3. 음수가 존재하면 안되는 곳에 음수값
4. IQR 기준 이상치
5. 파생컬럼 결측치

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 스타일 설정
plt.style.use('seaborn-v0_8-whitegrid')

# 색상 팔레트
COLORS = {
    'problem': '#E74C3C',      # 빨강 - 문제
    'normal': '#2ECC71',       # 초록 - 정상
    'warning': '#F39C12',      # 주황 - 경고
    'info': '#3498DB',         # 파랑 - 정보
    'highlight': '#9B59B6'     # 보라 - 강조
}

In [ ]:
# 데이터 로드
data_path = Path('../data/raw/기업신용평가정보_합성데이터.csv')

# CP949 (한국어 Windows) 인코딩으로 로드
df = pd.read_csv(data_path, encoding='cp949')

print(f"데이터 크기: {len(df):,}행 × {len(df.columns)}열")
print(f"\n컬럼 목록 (처음 20개):")
print(df.columns.tolist()[:20])

---
## 1. 중복 데이터 분석

In [ ]:
# 중복 데이터 분석
total_rows = len(df)
duplicate_rows = df.duplicated().sum()
unique_rows = total_rows - duplicate_rows

print(f"전체 행: {total_rows:,}")
print(f"중복 행: {duplicate_rows:,}")
print(f"고유 행: {unique_rows:,}")
print(f"중복 비율: {duplicate_rows/total_rows*100:.2f}%")

In [ ]:
# 중복 데이터 시각화
fig, ax = plt.subplots(figsize=(8, 6))

sizes = [unique_rows, duplicate_rows]
labels = [f'고유 데이터\n{unique_rows:,}건', f'중복 데이터\n{duplicate_rows:,}건']
colors = [COLORS['normal'], COLORS['problem']]
explode = (0, 0.1)  # 중복 데이터 강조

wedges, texts, autotexts = ax.pie(
    sizes, 
    labels=labels,
    colors=colors,
    explode=explode,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 12}
)

ax.set_title('1. 중복 데이터 현황', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../docs/eda_1_duplicates.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## 2. 경과일수 컬럼의 이상값 (999999999)

In [ ]:
# 경과일수 컬럼 찾기
# 컬럼명에 '경과' 또는 'D2B'가 포함된 컬럼
days_cols = [col for col in df.columns if '경과' in col or col.startswith('D2B')]
print(f"경과일수 관련 컬럼: {days_cols}")

# 999999999 값 분석
special_value = 999999999
days_analysis = {}

for col in days_cols:
    if col in df.columns:
        col_data = pd.to_numeric(df[col], errors='coerce')
        total = col_data.notna().sum()
        special_count = (col_data == special_value).sum()
        normal_count = total - special_count
        
        days_analysis[col] = {
            'total': total,
            'special': special_count,
            'normal': normal_count,
            'special_pct': special_count / total * 100 if total > 0 else 0
        }
        
        print(f"\n{col}:")
        print(f"  - 전체: {total:,}")
        print(f"  - 999999999 (이벤트 없음): {special_count:,} ({special_count/total*100:.1f}%)")
        print(f"  - 정상값: {normal_count:,}")

In [ ]:
# 경과일수 분포 시각화 (999999999 vs 정상값)
if days_analysis:
    fig, axes = plt.subplots(1, len(days_analysis), figsize=(6*len(days_analysis), 5))
    if len(days_analysis) == 1:
        axes = [axes]
    
    for ax, (col, stats) in zip(axes, days_analysis.items()):
        sizes = [stats['normal'], stats['special']]
        labels = [f"정상값\n{stats['normal']:,}건", f"999999999\n{stats['special']:,}건"]
        colors = [COLORS['normal'], COLORS['warning']]
        
        wedges, texts, autotexts = ax.pie(
            sizes,
            labels=labels,
            colors=colors,
            autopct='%1.1f%%',
            startangle=90,
            textprops={'fontsize': 10}
        )
        ax.set_title(f'{col}', fontsize=12, fontweight='bold')
    
    plt.suptitle('2. 경과일수 컬럼의 특수값 (999999999) 현황', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../docs/eda_2_special_values.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

In [ ]:
# 정상값의 분포 시각화 (999999999 제외)
if days_cols:
    col = days_cols[0]  # 첫 번째 경과일수 컬럼
    col_data = pd.to_numeric(df[col], errors='coerce')
    normal_data = col_data[(col_data != 999999999) & (col_data.notna())]
    
    if len(normal_data) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        
        # 히스토그램
        ax.hist(normal_data, bins=50, color=COLORS['info'], edgecolor='white', alpha=0.7)
        ax.axvline(x=90, color=COLORS['problem'], linestyle='--', linewidth=2, label='3개월 (90일)')
        ax.axvline(x=365, color=COLORS['warning'], linestyle='--', linewidth=2, label='1년 (365일)')
        ax.axvline(x=730, color=COLORS['normal'], linestyle='--', linewidth=2, label='2년 (730일)')
        
        ax.set_xlabel('경과일수', fontsize=12)
        ax.set_ylabel('기업 수', fontsize=12)
        ax.set_title(f'경과일수 분포 (999999999 제외)', fontsize=14, fontweight='bold')
        ax.legend()
        
        plt.tight_layout()
        plt.savefig('../docs/eda_2_days_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
        plt.show()

---
## 3. 음수값이 존재하면 안되는 컬럼의 음수값

In [ ]:
# 양수여야 할 컬럼 목록 (한국어 컬럼명)
positive_cols_map = {
    # 재무상태표 - 자산
    '자산총계': '자산총계',
    '유동자산': '유동자산', 
    '재고자산': '재고자산',
    '매출채권': '매출채권',
    '단기차입금': '단기차입금',
    '차입금': '차입금',
    '자본총계': '자본총계',
    
    # 손익계산서
    '매출액': '매출액',
    '매출원가': '매출원가',
    '판매비와관리비': '판매비와관리비',
}

# 음수값 분석
negative_analysis = {}

for col, name in positive_cols_map.items():
    if col in df.columns:
        col_data = pd.to_numeric(df[col], errors='coerce')
        total = col_data.notna().sum()
        negative_count = (col_data < 0).sum()
        
        if negative_count > 0:
            negative_analysis[name] = {
                'col': col,
                'total': total,
                'negative': negative_count,
                'negative_pct': negative_count / total * 100
            }

print("음수값이 발견된 컬럼:")
if negative_analysis:
    for name, stats in sorted(negative_analysis.items(), key=lambda x: x[1]['negative'], reverse=True):
        print(f"  - {name}: {stats['negative']:,}건 ({stats['negative_pct']:.2f}%)")
else:
    print("  (음수값이 발견된 컬럼 없음)")

In [ ]:
# 음수값 시각화
if negative_analysis:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    names = list(negative_analysis.keys())
    counts = [negative_analysis[name]['negative'] for name in names]
    pcts = [negative_analysis[name]['negative_pct'] for name in names]
    
    # 정렬
    sorted_data = sorted(zip(names, counts, pcts), key=lambda x: x[1], reverse=True)
    names, counts, pcts = zip(*sorted_data)
    
    bars = ax.barh(range(len(names)), counts, color=COLORS['problem'], edgecolor='white')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    
    # 값 표시
    for i, (count, pct) in enumerate(zip(counts, pcts)):
        ax.text(count + max(counts)*0.02, i, f'{count:,}건 ({pct:.1f}%)', 
                va='center', fontsize=10)
    
    ax.set_xlabel('음수값 개수', fontsize=12)
    ax.set_title('3. 양수여야 할 컬럼의 음수값 현황', fontsize=14, fontweight='bold')
    ax.set_xlim(0, max(counts) * 1.3)
    
    plt.tight_layout()
    plt.savefig('../docs/eda_3_negative_values.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("음수값이 발견된 컬럼이 없습니다.")

---
## 4. IQR 기준 이상치 분석

In [ ]:
def count_iqr_outliers(series, threshold=1.5):
    """IQR 방법으로 이상치 개수 계산"""
    clean_series = pd.to_numeric(series, errors='coerce').dropna()
    if len(clean_series) == 0:
        return 0, 0, 0
    
    Q1 = clean_series.quantile(0.25)
    Q3 = clean_series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    
    lower_outliers = (clean_series < lower_bound).sum()
    upper_outliers = (clean_series > upper_bound).sum()
    
    return lower_outliers, upper_outliers, len(clean_series)

# Leaf 컬럼 이상치 분석 (한국어 컬럼명)
leaf_cols_map = {
    '자산총계': '자산총계',
    '유동자산': '유동자산',
    '재고자산': '재고자산',
    '부채총계': '부채총계',
    '자본총계': '자본총계',
    '매출액': '매출액',
    '매출원가': '매출원가',
    '영업손익': '영업손익',
    '당기순이익': '당기순이익',
    '영업활동현금흐름': '영업활동현금흐름',
}

outlier_analysis = {}

for col, name in leaf_cols_map.items():
    if col in df.columns:
        lower, upper, total = count_iqr_outliers(df[col], threshold=1.5)
        total_outliers = lower + upper
        
        if total_outliers > 0:
            outlier_analysis[name] = {
                'col': col,
                'lower': lower,
                'upper': upper,
                'total_outliers': total_outliers,
                'total': total,
                'outlier_pct': total_outliers / total * 100 if total > 0 else 0
            }

print("IQR 1.5배 기준 이상치 현황:")
if outlier_analysis:
    for name, stats in sorted(outlier_analysis.items(), key=lambda x: x[1]['total_outliers'], reverse=True):
        print(f"  - {name}: {stats['total_outliers']:,}건 ({stats['outlier_pct']:.1f}%) [하한:{stats['lower']:,}, 상한:{stats['upper']:,}]")
else:
    print("  (이상치 감지된 컬럼 없음)")

In [ ]:
# 이상치 시각화
if outlier_analysis:
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # 정렬
    sorted_items = sorted(outlier_analysis.items(), key=lambda x: x[1]['total_outliers'], reverse=True)
    names = [item[0] for item in sorted_items]
    lower_counts = [item[1]['lower'] for item in sorted_items]
    upper_counts = [item[1]['upper'] for item in sorted_items]
    
    x = np.arange(len(names))
    width = 0.35
    
    bars1 = ax.barh(x - width/2, lower_counts, width, label='하한 이상치', color=COLORS['info'])
    bars2 = ax.barh(x + width/2, upper_counts, width, label='상한 이상치', color=COLORS['problem'])
    
    ax.set_yticks(x)
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel('이상치 개수', fontsize=12)
    ax.set_title('4. IQR 기준 이상치 현황 (IQR × 1.5)', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    
    # 총 이상치 수 표시
    for i, (item) in enumerate(sorted_items):
        total = item[1]['total_outliers']
        pct = item[1]['outlier_pct']
        max_val = max(item[1]['lower'], item[1]['upper'])
        ax.text(max_val + 100, i, f'{total:,}건 ({pct:.1f}%)', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../docs/eda_4_outliers.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

In [ ]:
# 박스플롯으로 이상치 시각화 (상위 6개 컬럼)
top_cols = list(outlier_analysis.keys())[:6]
top_col_codes = [outlier_analysis[name]['col'] for name in top_cols]

if top_cols:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    
    for ax, name, col in zip(axes, top_cols, top_col_codes):
        col_data = pd.to_numeric(df[col], errors='coerce').dropna()
        
        # 박스플롯
        bp = ax.boxplot(col_data, vert=True, patch_artist=True)
        bp['boxes'][0].set_facecolor(COLORS['info'])
        bp['boxes'][0].set_alpha(0.7)
        
        # 이상치 강조
        for flier in bp['fliers']:
            flier.set(marker='o', color=COLORS['problem'], alpha=0.5, markersize=3)
        
        ax.set_title(f'{name}', fontsize=11, fontweight='bold')
        ax.set_ylabel('값')
        ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    
    plt.suptitle('주요 컬럼 박스플롯 (이상치 분포)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../docs/eda_4_boxplots.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

---
## 5. 파생컬럼 결측치 분석

In [ ]:
# 파생컬럼 (재무비율) 결측치 분석
derived_cols_map = {
    'R006': '부채비율',
    'R007': '자기자본비율',
    'R008': '유동비율',
    'R012': '차입금의존도',
    'R013': '매출원가율',
    'R015': '영업이익률',
    'R016': '당기순이익률',
    'R018': 'ROE',
    'R019': '매출채권회전율',
    'R020': '재고자산회전율',
    'R022': '총자산회전율',
    'R023': 'ROA',
    'N001': '단기차입금의존도',
    'N003': '순운전자본회전율',
    'N005': '매출총이익률',
}

missing_analysis = {}
total_rows = len(df)

for col, name in derived_cols_map.items():
    if col in df.columns:
        missing_count = df[col].isna().sum()
        missing_pct = missing_count / total_rows * 100
        
        missing_analysis[name] = {
            'col': col,
            'missing': missing_count,
            'missing_pct': missing_pct,
            'valid': total_rows - missing_count
        }

print("파생컬럼 결측치 현황:")
for name, stats in sorted(missing_analysis.items(), key=lambda x: x[1]['missing'], reverse=True):
    print(f"  - {name} ({stats['col']}): {stats['missing']:,}건 ({stats['missing_pct']:.1f}%)")

In [ ]:
# 파생컬럼 결측치 시각화
if missing_analysis:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # 정렬
    sorted_items = sorted(missing_analysis.items(), key=lambda x: x[1]['missing_pct'], reverse=True)
    names = [item[0] for item in sorted_items]
    missing_pcts = [item[1]['missing_pct'] for item in sorted_items]
    valid_pcts = [100 - pct for pct in missing_pcts]
    
    y = np.arange(len(names))
    
    # 스택 바 차트
    bars1 = ax.barh(y, valid_pcts, label='유효 데이터', color=COLORS['normal'])
    bars2 = ax.barh(y, missing_pcts, left=valid_pcts, label='결측치', color=COLORS['problem'])
    
    ax.set_yticks(y)
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel('비율 (%)', fontsize=12)
    ax.set_title('5. 파생컬럼(재무비율) 결측치 현황', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_xlim(0, 100)
    
    # 결측치 비율 표시
    for i, (name, stats) in enumerate(sorted_items):
        if stats['missing_pct'] > 1:  # 1% 이상만 표시
            ax.text(100 - stats['missing_pct']/2, i, f"{stats['missing_pct']:.1f}%", 
                   va='center', ha='center', fontsize=9, color='white', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../docs/eda_5_missing_derived.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

---
## 종합 요약: 데이터 품질 문제 대시보드

In [ ]:
# 종합 요약 시각화
fig = plt.figure(figsize=(16, 10))

# 변수 기본값 설정 (데이터에 문제가 없는 경우)
total_negative = sum(stats['negative'] for stats in negative_analysis.values()) if negative_analysis else 0
total_outliers = sum(stats['total_outliers'] for stats in outlier_analysis.values()) if outlier_analysis else 0
avg_missing_pct = np.mean([stats['missing_pct'] for stats in missing_analysis.values()]) if missing_analysis else 0
special_count = days_analysis[list(days_analysis.keys())[0]]['special'] if days_analysis else 0

# 1. 중복 데이터 (왼쪽 상단)
ax1 = fig.add_subplot(2, 3, 1)
sizes = [unique_rows, duplicate_rows] if duplicate_rows > 0 else [unique_rows, 1]
colors = [COLORS['normal'], COLORS['problem']]
ax1.pie(sizes, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title(f'1. 중복 데이터\n({duplicate_rows:,}건)', fontsize=11, fontweight='bold')

# 2. 특수값 (중앙 상단)
ax2 = fig.add_subplot(2, 3, 2)
if days_analysis:
    first_col = list(days_analysis.keys())[0]
    stats = days_analysis[first_col]
    sizes = [stats['normal'], stats['special']]
    colors = [COLORS['normal'], COLORS['warning']]
    ax2.pie(sizes, colors=colors, autopct='%1.1f%%', startangle=90)
    ax2.set_title(f'2. 경과일수 특수값\n(999999999: {stats["special"]:,}건)', fontsize=11, fontweight='bold')
else:
    ax2.text(0.5, 0.5, '경과일수 컬럼 없음', ha='center', va='center')
    ax2.set_title('2. 경과일수 특수값', fontsize=11, fontweight='bold')

# 3. 음수값 (오른쪽 상단)
ax3 = fig.add_subplot(2, 3, 3)
if negative_analysis:
    names = list(negative_analysis.keys())[:5]
    counts = [negative_analysis[name]['negative'] for name in names]
    ax3.barh(range(len(names)), counts, color=COLORS['problem'])
    ax3.set_yticks(range(len(names)))
    ax3.set_yticklabels(names, fontsize=9)
    ax3.invert_yaxis()
    ax3.set_title(f'3. 음수값 (상위 5개)\n(총 {total_negative:,}건)', fontsize=11, fontweight='bold')
else:
    ax3.text(0.5, 0.5, '음수값 없음\n(정상)', ha='center', va='center', fontsize=12, color=COLORS['normal'])
    ax3.set_title('3. 음수값\n(0건)', fontsize=11, fontweight='bold')

# 4. 이상치 (왼쪽 하단)
ax4 = fig.add_subplot(2, 3, 4)
if outlier_analysis:
    sorted_items = sorted(outlier_analysis.items(), key=lambda x: x[1]['total_outliers'], reverse=True)[:5]
    names = [item[0] for item in sorted_items]
    counts = [item[1]['total_outliers'] for item in sorted_items]
    ax4.barh(range(len(names)), counts, color=COLORS['warning'])
    ax4.set_yticks(range(len(names)))
    ax4.set_yticklabels(names, fontsize=9)
    ax4.invert_yaxis()
    ax4.set_title(f'4. IQR 이상치 (상위 5개)\n(총 {total_outliers:,}건)', fontsize=11, fontweight='bold')
else:
    ax4.text(0.5, 0.5, '이상치 없음', ha='center', va='center')
    ax4.set_title('4. IQR 이상치', fontsize=11, fontweight='bold')

# 5. 파생컬럼 결측치 (중앙 하단)
ax5 = fig.add_subplot(2, 3, 5)
if missing_analysis:
    sorted_items = sorted(missing_analysis.items(), key=lambda x: x[1]['missing_pct'], reverse=True)[:5]
    names = [item[0] for item in sorted_items]
    pcts = [item[1]['missing_pct'] for item in sorted_items]
    ax5.barh(range(len(names)), pcts, color=COLORS['highlight'])
    ax5.set_yticks(range(len(names)))
    ax5.set_yticklabels(names, fontsize=9)
    ax5.invert_yaxis()
    ax5.set_xlabel('결측률 (%)')
    ax5.set_title(f'5. 파생컬럼 결측치 (상위 5개)\n(평균 {avg_missing_pct:.1f}%)', fontsize=11, fontweight='bold')
else:
    ax5.text(0.5, 0.5, '파생컬럼 없음', ha='center', va='center')
    ax5.set_title('5. 파생컬럼 결측치', fontsize=11, fontweight='bold')

# 6. 요약 텍스트 (오른쪽 하단)
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')

summary_text = f"""
데이터 품질 문제 요약
━━━━━━━━━━━━━━━━━━━━━━━━

전체 데이터: {total_rows:,}건

1. 중복 데이터: {duplicate_rows:,}건
2. 특수값(999999999): {special_count:,}건
3. 음수값: {total_negative:,}건
4. IQR 이상치: {total_outliers:,}건
5. 파생컬럼 결측: 평균 {avg_missing_pct:.1f}%

━━━━━━━━━━━━━━━━━━━━━━━━
→ 8단계 정제 파이프라인 필요
"""
ax6.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center',
         fontfamily='AppleGothic', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('EDA: 데이터 품질 문제 종합 대시보드', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/eda_summary_dashboard.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
print("\n" + "="*60)
print("EDA 시각화 완료!")
print("="*60)
print("\n생성된 이미지 파일:")
print("  - docs/eda_1_duplicates.png       : 중복 데이터 현황")
print("  - docs/eda_2_special_values.png   : 경과일수 특수값 현황")
print("  - docs/eda_2_days_distribution.png: 경과일수 분포")
print("  - docs/eda_3_negative_values.png  : 음수값 현황")
print("  - docs/eda_4_outliers.png         : IQR 이상치 현황")
print("  - docs/eda_4_boxplots.png         : 박스플롯")
print("  - docs/eda_5_missing_derived.png  : 파생컬럼 결측치")
print("  - docs/eda_summary_dashboard.png  : 종합 대시보드")